# Master Pipeline

This is the consolidation notebook mentioned in `05_video_inference.ipynb`'s setup cell: "the clean way to share logic across notebooks is... converting shared logic into a proper .py module... (which we'll actually do when we build the master notebook)".

`pipeline_core.py` now holds the shared logic (`DeepfakeDetectionPipeline` class) that `04_inference.ipynb`, `05_video_inference.ipynb`, and the old gradio-app notebook (since removed) used to duplicate three separate times. This notebook imports it once and exposes Image / Video / Webcam through a single Gradio app.

**Before running this notebook:** place `pipeline_core.py` (and `audio_deepfake_module.py`, if you're using the audio branch) in the same folder as your notebooks.

**On top of consolidation, this adds:**
- Grad-CAM region localization (shows which part of the face drove the decision)
- Grounded explanations (the LLM gets concrete signals -- confidence margin, Grad-CAM region, frame agreement -- instead of just prediction+confidence, which is what was producing generic-sounding text before)
- Optional 3-class face-swap-aware prediction if you point it at `efficientnet_faceswap_best.pth` from `07_faceswap_training.ipynb` instead of the original binary checkpoint

Step 1 - Setup

Choose which face model to load:
- **Binary** (`efficientnet_best.pth`, `class_names=['fake','real']`) -- your current production model
- **Face-swap-aware** (`efficientnet_faceswap_best.pth`, `class_names=['faceswap','other_fake','real']`) -- from `07_faceswap_training.ipynb`, once trained

Just flip `USE_FACESWAP_MODEL` below once you've trained and are ready to switch over.

In [ ]:
import os
from config import BASE_DIR, DATASETS_DIR, SAVED_MODELS_DIR, OUTPUTS_DIR
from pipeline_core import DeepfakeDetectionPipeline, BINARY_CLASS_NAMES, FACESWAP_CLASS_NAMES
from dotenv import load_dotenv

load_dotenv(str(BASE_DIR / ".env"))
HF_TOKEN = os.getenv("HF_TOKEN")

USE_FACESWAP_MODEL = True  # flip to True once 07_faceswap_training.ipynb has produced a checkpoint

if USE_FACESWAP_MODEL:
    face_checkpoint = str(SAVED_MODELS_DIR / "efficientnet_faceswap_best.pth")
    class_names = FACESWAP_CLASS_NAMES
else:
    face_checkpoint = str(SAVED_MODELS_DIR / "efficientnet_best.pth")
    class_names = BINARY_CLASS_NAMES

audio_checkpoint = str(SAVED_MODELS_DIR / "asvspoof_efficientnet_best.pth")  # optional; only loaded if the file exists

pipeline = DeepfakeDetectionPipeline(
    face_checkpoint=face_checkpoint,
    face_class_names=class_names,
    audio_checkpoint=audio_checkpoint if os.path.exists(audio_checkpoint) else None,
    hf_token=HF_TOKEN,
)

print("Pipeline ready. Class names:", pipeline.class_names)
print("Audio branch loaded:", pipeline.audio_model is not None)

Step 2 - Quick sanity check on a known test image

Same idea as the quick test in `04_inference.ipynb` Step 3, just going through the consolidated pipeline instead.

In [ ]:
test_dir = str(DATASETS_DIR / "test" / "real")
test_image_path = os.path.join(test_dir, os.listdir(test_dir)[0])

result = pipeline.analyze_image(test_image_path)
print("Prediction:", result["prediction"], f"({result['confidence']:.1f}%)")
print("Probabilities:", result["probabilities"])
print("Top Grad-CAM regions:", result["top_regions"])
if "explanation" in result:
    print("\nExplanation:\n", result["explanation"])

Step 3 - Gradio app (Image / Video / Webcam), all backed by the one pipeline object

Same three tabs as before, but every tab now calls `pipeline.analyze_*()` instead of each having its own copy of face-crop + model-load + explanation logic. This is the fix for the "preprocessing must match between train/inference" pitfall noted in your own devlog -- there's only one preprocessing path now, used everywhere.

In [ ]:
import gradio as gr
gr.close_all()

def image_tab_fn(pil_image):
    if pil_image is None:
        return "Please upload an image."
    tmp_path = str(OUTPUTS_DIR / "temp_master_upload.jpg")
    pil_image.save(tmp_path)
    result = pipeline.analyze_image(tmp_path)

    lines = [f"Prediction: {result['prediction'].upper()} ({result['confidence']:.1f}%)", ""]
    for cls, p in result["probabilities"].items():
        lines.append(f"  {cls}: {p*100:.1f}%")
    if result.get("top_regions"):
        lines.append("")
        lines.append("Model attention (Grad-CAM): " + ", ".join(f"{n} ({w*100:.0f}%)" for n, w in result["top_regions"]))
    if "explanation" in result:
        lines.append("")
        lines.append("Explanation:")
        lines.append(result["explanation"])
    return "\n".join(lines)


def video_tab_fn(video_path):
    if video_path is None:
        return "Please upload a video."
    result = pipeline.analyze_video(video_path)
    if "error" in result:
        return result["error"]

    lines = [f"Prediction: {result['prediction'].upper()} ({result['confidence']:.1f}%)",
              f"(Averaged across sampled frames; majority vote: {result['majority_vote']}, "
              f"frame agreement: {result['frame_agreement']*100:.0f}%)", ""]
    for cls, p in result["probabilities"].items():
        lines.append(f"  {cls}: {p*100:.1f}%")
    if result.get("top_regions"):
        lines.append("")
        lines.append("Model attention (Grad-CAM, last sampled frame): " +
                      ", ".join(f"{n} ({w*100:.0f}%)" for n, w in result["top_regions"]))
    if "audio_prediction" in result:
        lines.append("")
        lines.append(f"Audio verdict: {result['audio_prediction'].upper()}")
        lines.append(f"Overall: {result['overall_verdict']}")
    if "explanation" in result:
        lines.append("")
        lines.append("Explanation:")
        lines.append(result["explanation"])
    return "\n".join(lines)


def webcam_tab_fn(pil_image):
    if pil_image is None:
        return "No frame captured -- click the camera icon, then Analyze."
    import numpy as np
    import cv2
    img_array = cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)
    result = pipeline.analyze_frame(img_array, explain=False)  # no LLM call per frame -- keeps the live loop fast
    return f"Prediction: {result['prediction'].upper()} ({result['confidence']:.1f}%)"


with gr.Blocks(title="AI-Powered Deepfake Detection System") as demo:
    gr.Markdown("# AI-Powered Deepfake Detection System (Master Pipeline)")
    gr.Markdown("Detect deepfakes in images, videos, or live webcam feed -- one shared backend, grounded explanations, Grad-CAM region localization.")

    with gr.Tab("Image"):
        img_input = gr.Image(type="pil", label="Upload an image")
        img_button = gr.Button("Analyze Image")
        img_output = gr.Textbox(label="Result", lines=12)
        img_button.click(fn=image_tab_fn, inputs=img_input, outputs=img_output)

    with gr.Tab("Video"):
        vid_input = gr.Video(label="Upload a video")
        vid_button = gr.Button("Analyze Video")
        vid_output = gr.Textbox(label="Result", lines=14)
        vid_button.click(fn=video_tab_fn, inputs=vid_input, outputs=vid_output)

    with gr.Tab("Webcam"):
        gr.Markdown("Click the camera icon, then Analyze Snapshot.")
        webcam_input = gr.Image(type="pil", label="Webcam", sources=["webcam"])
        webcam_button = gr.Button("Analyze Snapshot", variant="primary")
        webcam_output = gr.Textbox(label="Result")
        webcam_button.click(fn=webcam_tab_fn, inputs=webcam_input, outputs=webcam_output)

demo.launch()